# KG1 Nemotron - Pre-score Calibration

Objetivo: calibrar nosso gate local contra scores reais do Kaggle.

Este notebook:

1. instala dependencias;
2. monta Google Drive;
3. carrega os scripts do pack `kg1_prescore_calibration_pack.zip`;
4. descobre adapters em uma pasta do Drive;
5. roda pre-score em GPU para cada adapter;
6. cruza os resultados locais brutos com scores reais conhecidos;
7. gera uma curva `local_score -> Kaggle public score`.

Ele **nao faz submit**.

In [6]:
!nvidia-smi
!python --version
!pip -q install -U "transformers>=4.52.0" "peft>=0.15.0" accelerate bitsandbytes safetensors huggingface_hub sentencepiece protobuf pandas tqdm kaggle

Wed Apr 29 16:03:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   28C    P0             70W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [7]:
import os, json, shutil, zipfile, textwrap
from pathlib import Path

try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    def secret(name, default=""):
        try:
            return userdata.get(name) or default
        except Exception:
            return default
except Exception:
    userdata = None
    def secret(name, default=""):
        return os.environ.get(name, default)

HF_TOKEN = secret("HF_TOKEN", os.environ.get("HF_TOKEN", ""))
KAGGLE_USERNAME = secret("KAGGLE_USERNAME", os.environ.get("KAGGLE_USERNAME", ""))
KAGGLE_KEY = secret("KAGGLE_KEY", os.environ.get("KAGGLE_KEY", ""))

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
if KAGGLE_USERNAME and KAGGLE_KEY:
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"] = KAGGLE_KEY

WORK_DIR = Path("/content/kg1_prescore")
DRIVE_DIR = Path("/content/drive/MyDrive/KG1_Nemotron_prescore")
PACK_ZIP = DRIVE_DIR / "kg1_prescore_calibration_pack.zip"
ADAPTERS_ROOT = DRIVE_DIR / "adapters"
OUT_DIR = DRIVE_DIR / "prescore_outputs"

WORK_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
ADAPTERS_ROOT.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("WORK_DIR:", WORK_DIR)
print("DRIVE_DIR:", DRIVE_DIR)
print("PACK_ZIP exists:", PACK_ZIP.exists(), PACK_ZIP)
print("ADAPTERS_ROOT:", ADAPTERS_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
WORK_DIR: /content/kg1_prescore
DRIVE_DIR: /content/drive/MyDrive/KG1_Nemotron_prescore
PACK_ZIP exists: False /content/drive/MyDrive/KG1_Nemotron_prescore/kg1_prescore_calibration_pack.zip
ADAPTERS_ROOT: /content/drive/MyDrive/KG1_Nemotron_prescore/adapters


## Como preparar os adapters

Coloque os adapters ou `submission.zip` já conhecidos dentro desta pasta do Drive:

`/content/drive/MyDrive/KG1_Nemotron_prescore/adapters`

Estruturas aceitas:

- `adapters/canonical_086/adapter_config.json` + `adapter_model.safetensors`
- `adapters/v088_084/submission.zip`
- `adapters/qualquer_nome/submission.zip`

O notebook extrai zips automaticamente para subpastas.

In [8]:
if not PACK_ZIP.exists():
    print("Pack nao encontrado no Drive; baixando fallback do Hugging Face...")
    from huggingface_hub import hf_hub_download
    downloaded_pack = hf_hub_download(
        repo_id="felipesp1983/kg1-nemotron-training",
        repo_type="dataset",
        filename="prescore/kg1_prescore_calibration_pack.zip",
        token=HF_TOKEN or None,
    )
    shutil.copy2(downloaded_pack, PACK_ZIP)
    print("Pack baixado e salvo em:", PACK_ZIP)

assert PACK_ZIP.exists(), f"Pack ausente: {PACK_ZIP}"
with zipfile.ZipFile(PACK_ZIP) as zf:
    zf.extractall(WORK_DIR)

print("Pack extraido.")
!find /content/kg1_prescore -maxdepth 3 -type f | head -80

Pack nao encontrado no Drive; baixando fallback do Hugging Face...


prescore/kg1_prescore_calibration_pack.z(…):   0%|          | 0.00/544k [00:00<?, ?B/s]

Pack baixado e salvo em: /content/drive/MyDrive/KG1_Nemotron_prescore/kg1_prescore_calibration_pack.zip
Pack extraido.
/content/kg1_prescore/data/v94/v94_equation_crypt_manifest.json
/content/kg1_prescore/data/v94/v94_equation_crypt_val.jsonl
/content/kg1_prescore/data/v90/v90_val_gold_safe_stratified.jsonl
/content/kg1_prescore/data/v100/v100_programmatic_repair_manifest.json
/content/kg1_prescore/data/v100/v100_programmatic_repair_val.jsonl
/content/kg1_prescore/scripts/kg1_local_metric_gate.py
/content/kg1_prescore/scripts/batch_validate_prescore.py
/content/kg1_prescore/scripts/kg1_kaggle_like_gate.py
/content/kg1_prescore/scripts/validate_prescore_gate_calibration.py
/content/kg1_prescore/scripts/kg1_submission_gate.py
/content/kg1_prescore/scripts/kg1_behavioral_submission_gate.py


In [9]:
# Extrai submission.zip dentro de ADAPTERS_ROOT para diretorios utilizaveis pelo PEFT.
for zip_path in ADAPTERS_ROOT.rglob("*.zip"):
    out = zip_path.with_suffix("")
    out.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        if "adapter_config.json" in names and "adapter_model.safetensors" in names:
            zf.extract("adapter_config.json", out)
            zf.extract("adapter_model.safetensors", out)
            print("extraido:", zip_path, "->", out)

adapter_dirs = []
for cfg in ADAPTERS_ROOT.rglob("adapter_config.json"):
    if (cfg.parent / "adapter_model.safetensors").exists():
        adapter_dirs.append(cfg.parent)

print("Adapters encontrados:", len(adapter_dirs))
for p in adapter_dirs[:50]:
    print("-", p)

Adapters encontrados: 0


In [10]:
# Crie/ajuste os pares conhecidos de calibracao.
# Use caminhos EXATOS dos diretórios de adapter impressos acima.
CALIBRATION_PAIRS = {
    # Exemplo:
    # "/content/drive/MyDrive/KG1_Nemotron_prescore/adapters/canonical_086": 0.86,
    # "/content/drive/MyDrive/KG1_Nemotron_prescore/adapters/v088_084/submission": 0.84,
}

calibration_path = OUT_DIR / "calibration_pairs.json"
calibration_path.write_text(json.dumps(CALIBRATION_PAIRS, indent=2, sort_keys=True), encoding="utf-8")
print(calibration_path)
print(json.dumps(CALIBRATION_PAIRS, indent=2, sort_keys=True))

/content/drive/MyDrive/KG1_Nemotron_prescore/prescore_outputs/calibration_pairs.json
{}


In [11]:
# Dataset holdout principal para pre-score.
# Opcoes incluidas no pack:
# - data/v94/v94_equation_crypt_val.jsonl: foca equation/crypt + rehearsal.
# - data/v100/v100_programmatic_repair_val.jsonl: foca bit/equation + rehearsal.
# - data/v90/v90_val_gold_safe_stratified.jsonl: holdout estratificado antigo.
DATASET_PATH = WORK_DIR / "data" / "v94" / "v94_equation_crypt_val.jsonl"
assert DATASET_PATH.exists(), DATASET_PATH
print(DATASET_PATH, "lines=", sum(1 for _ in DATASET_PATH.open(encoding="utf-8")))

/content/kg1_prescore/data/v94/v94_equation_crypt_val.jsonl lines= 279


In [12]:
# Rodada curta: 30-60 exemplos para smoke. Rodada séria: 120-300 exemplos.
# score-mode raw é obrigatório para alegação Kaggle-like; canonical é diagnóstico.
N_SAMPLES = 60
MAX_ADAPTERS = 8
MAX_NEW_TOKENS = 1024
SCORE_MODE = "raw"

cmd = f'''
python /content/kg1_prescore/scripts/batch_validate_prescore.py \
  --adapters-root "{ADAPTERS_ROOT}" \
  --dataset-path "{DATASET_PATH}" \
  --calibration-pairs "{calibration_path}" \
  --n-samples {N_SAMPLES} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --score-mode {SCORE_MODE} \
  --output-dir "{OUT_DIR}" \
  --max-adapters {MAX_ADAPTERS}
'''
print(cmd)


python /content/kg1_prescore/scripts/batch_validate_prescore.py   --adapters-root "/content/drive/MyDrive/KG1_Nemotron_prescore/adapters"   --dataset-path "/content/kg1_prescore/data/v94/v94_equation_crypt_val.jsonl"   --calibration-pairs "/content/drive/MyDrive/KG1_Nemotron_prescore/prescore_outputs/calibration_pairs.json"   --n-samples 60   --max-new-tokens 1024   --score-mode raw   --output-dir "/content/drive/MyDrive/KG1_Nemotron_prescore/prescore_outputs"   --max-adapters 8



In [13]:
# Execute quando os adapters e calibration_pairs estiverem prontos.
!{cmd}

=== Step 1: Discovering adapters ===
Found 0 adapters:
[FAIL] No adapters found


In [14]:
# Inspeciona saidas.
import json
from pathlib import Path
for path in sorted(OUT_DIR.glob("*.json")):
    print("\n###", path.name)
    txt = path.read_text(encoding="utf-8", errors="replace")
    print(txt[:4000])


### calibration_pairs.json
{}


## Regras de uso do resultado

Nao use o score local bruto como score Kaggle.

So aceite o gate se:

- houver pelo menos 3 pares exatos adapter->Kaggle;
- o erro medio de calibracao for baixo;
- o canonical `0.86` for reproduzido perto de `0.86`;
- v088/Huikang forem corretamente classificados como `0.84` ou regressivos;
- o novo candidato bater o canonical localmente nas familias fracas sem perder nas fortes.